In [1]:
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/behavior-aware-bms"

In [4]:
!find /content/drive/MyDrive/behavior-aware-bms -name "*health*" -type f

/content/drive/MyDrive/behavior-aware-bms/data/features/battery_health_index_v1.csv
/content/drive/MyDrive/behavior-aware-bms/notebooks/06_battery_health_index.ipynb
/content/drive/MyDrive/behavior-aware-bms/docs/weighted_health_index.md
/content/drive/MyDrive/behavior-aware-bms/reports/metrics/health_index_distribution.csv


In [7]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/behavior-aware-bms"

import pandas as pd

battery = pd.read_csv(
    f"{PROJECT_ROOT}/data/features/battery_health_index_v1.csv"
)

print(battery.shape)
print(battery.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(34, 18)
['battery_id', 'avg_stress', 'avg_temp', 'fast_charge_duration', 'deep_discharge_duration', 'high_temp_duration', 'aggressive_discharge_count', 'avg_soc', 'aging_budget', 'stress_norm', 'temp_norm', 'dd_norm', 'fc_norm', 'soc_norm', 'health_index', 'battery_state', 'remaining_health', 'consumed_life']


In [8]:
battery["equivalent_aging_factor"] = (
      0.40*(battery["health_index"]/100)
    + 0.25*(battery["avg_temp"]/50)
    + 0.20*(battery["deep_discharge_duration"] /
            battery["deep_discharge_duration"].max())
    + 0.15*(battery["fast_charge_duration"] /
            battery["fast_charge_duration"].max())
)

In [9]:
battery["equivalent_aging_factor"] = (
    battery["equivalent_aging_factor"]
    .clip(0.05,1.0)
)

battery[
    ["battery_id","equivalent_aging_factor"]
].head()

,battery_id,equivalent_aging_factor
0,B0005,0.545425
1,B0006,0.520900
2,B0007,0.366672
3,B0018,0.476063
4,B0025,0.437970


In [10]:
battery["estimated_total_cycles"] = (
    1000 /
    battery["equivalent_aging_factor"]
)

In [11]:
battery["rul_cycles"] = (
    battery["estimated_total_cycles"] *
    battery["remaining_health"]/100
)

battery["rul_cycles"] = (
    battery["rul_cycles"]
    .round()
    .astype(int)
)

In [12]:
def replacement_policy(x):

    if x < 100:
        return "REPLACE"

    elif x < 300:
        return "PLAN_SERVICE"

    elif x < 600:
        return "MONITOR"

    else:
        return "NORMAL"


battery["replacement_policy"] = (
    battery["rul_cycles"]
    .apply(replacement_policy)
)

In [13]:
rul_dist = (
    battery["replacement_policy"]
    .value_counts()
)

print(rul_dist)

replacement_policy
NORMAL          27
PLAN_SERVICE     5
MONITOR          2
Name: count, dtype: int64


In [14]:
battery[
    [
        "battery_id",
        "battery_state",
        "remaining_health",
        "rul_cycles",
        "replacement_policy"
    ]
].sort_values(
    "rul_cycles"
).head(20)

,battery_id,battery_state,remaining_health,rul_cycles,replacement_policy
8,B0029,CRITICAL,15,245,PLAN_SERVICE
9,B0030,CRITICAL,15,246,PLAN_SERVICE
15,B0038,CRITICAL,15,261,PLAN_SERVICE
11,B0032,CRITICAL,15,262,PLAN_SERVICE
10,B0031,CRITICAL,15,264,PLAN_SERVICE
17,B0040,DEGRADED,25,496,MONITOR
16,B0039,DEGRADED,25,499,MONITOR
0,B0005,DEGRADED,35,642,NORMAL
1,B0006,DEGRADED,35,672,NORMAL
12,B0033,DEGRADED,35,706,NORMAL


In [15]:
print(rul_dist)

replacement_policy
NORMAL          27
PLAN_SERVICE     5
MONITOR          2
Name: count, dtype: int64


In [16]:
print(
    battery["rul_cycles"]
    .describe()
)

count      34.000000
mean     1180.088235
std       816.665629
min       245.000000
25%       680.500000
50%       967.000000
75%      1511.500000
max      3208.000000
Name: rul_cycles, dtype: float64
